# T07: Data Migration

This tutorial walks through the full data migration workflow: diff two schema
sources, create a migration pathway, submit a batch migration job, and poll
for job completion.

**Services required**: backend (`http://localhost:8002`) + migration-api (`http://localhost:8004`)

**Est. time**: 15 min

In [ ]:
# Cell 2 — service availability check for BOTH services
import os
import time

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
MIGRATION_URL = os.getenv("MIGRATION_URL", "http://localhost:8004")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
MIGRATION_HEADERS = {"Authorization": f"Bearer {API_KEY}"}

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

try:
    httpx.get(f"{MIGRATION_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Migration API available at {MIGRATION_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Migration API unavailable: {_e}")

## 1. Schema Diff — What Changed?

The migration API's `/diff` endpoint compares two schema sources and returns
a summary of added, removed, and changed elements.

In [ ]:
# Fetch first two source IDs from the backend
sources_resp = httpx.get(
    f"{BACKEND_URL}/api/v1/sources/",
    headers=HEADERS,
    params={"limit": 5},
    timeout=5.0,
)
assert sources_resp.status_code == 200
sources = sources_resp.json()["items"]

if len(sources) < 2:
    import pytest

    pytest.skip("Fewer than 2 sources found — run T02 (02_ingest_schemas.ipynb) first")

src_a_id = sources[0]["id"]
src_b_id = sources[1]["id"]
print(f"Diffing '{sources[0]['name']}' → '{sources[1]['name']}'")

diff_resp = httpx.post(
    f"{MIGRATION_URL}/api/v1/diff",
    headers=MIGRATION_HEADERS,
    json={"source_schema_id": src_a_id, "target_schema_id": src_b_id},
    timeout=30.0,
)
assert diff_resp.status_code == 200, f"Diff failed: {diff_resp.status_code}: {diff_resp.text}"
diff = diff_resp.json()
print(f"Added:   {len(diff.get('added', []))}")
print(f"Removed: {len(diff.get('removed', []))}")
print(f"Changed: {len(diff.get('changed', []))}")

## 2. Create a Migration Pathway

A migration pathway captures the intent to migrate data from one schema version
to another. Steps define the transformation rules for each changed element.

In [ ]:
pathway_resp = httpx.post(
    f"{MIGRATION_URL}/api/v1/pathways",
    headers=MIGRATION_HEADERS,
    json={
        "name": "tutorial-pathway",
        "source_schema_id": src_a_id,
        "target_schema_id": src_b_id,
        "steps": [],
    },
    timeout=10.0,
)
assert pathway_resp.status_code in (200, 201), (
    f"Create pathway failed: {pathway_resp.status_code}: {pathway_resp.text}"
)
pathway = pathway_resp.json()
pathway_id = pathway["id"]
print(f"Created pathway: {pathway_id}")
print(f"  name:   {pathway.get('name')}")
print(f"  steps:  {len(pathway.get('steps', []))}")

## 3. Submit a Batch Migration Job

The `/migrate` endpoint accepts a pathway ID and a list of records to migrate.
For async migrations, it returns a job ID that you can poll.

In [ ]:
migrate_resp = httpx.post(
    f"{MIGRATION_URL}/api/v1/migrate",
    headers=MIGRATION_HEADERS,
    json={
        "pathway_id": pathway_id,
        "records": [{"id": "test-001", "subject_name": "test"}],
    },
    timeout=30.0,
)
assert migrate_resp.status_code in (200, 201, 202), (
    f"Migrate failed: {migrate_resp.status_code}: {migrate_resp.text}"
)
job = migrate_resp.json()
job_id = job.get("id") or job.get("job_id")
print(f"Migration job: {job_id}")
print(f"  status: {job.get('status', '?')}")

## 4. Query Job Status

For async jobs, poll the `/jobs/{id}` endpoint until the job completes or
a timeout is reached.

In [ ]:
if job_id:
    for attempt in range(3):
        status_resp = httpx.get(
            f"{MIGRATION_URL}/api/v1/jobs/{job_id}",
            headers=MIGRATION_HEADERS,
            timeout=10.0,
        )
        if status_resp.status_code == 200:
            status = status_resp.json()
            print(f"Job {job_id}: status={status.get('status', '?')}")
            if status.get("status") in ("complete", "done", "finished", "success"):
                break
            time.sleep(2)
        else:
            print(f"Poll attempt {attempt + 1}: {status_resp.status_code}")
            break
else:
    print("No job ID returned — migration may be synchronous")

## Cleanup

In [ ]:
del_resp = httpx.delete(
    f"{MIGRATION_URL}/api/v1/pathways/{pathway_id}",
    headers=MIGRATION_HEADERS,
    timeout=5.0,
)
print(f"Deleted pathway {pathway_id}: {del_resp.status_code}")
print("✓ Cleanup complete")

## Congratulations!

You've completed all 7 undata tutorials:

1. [T01: Getting Started](01_getting_started.ipynb) — health check, sources, elements, auth
2. [T02: Ingest Schemas via CLI](02_ingest_schemas.ipynb) — BIDS and DANDI ingestion
3. [T03: Browse and Search Elements](03_browse_elements.ipynb) — pagination, filtering, history
4. [T04: Schema Classes and Element Mappings](04_mappings_aliases.ipynb) — mappings and aliases
5. [T05: LinkML Schema Export](05_linkml_export.ipynb) — generate-schema CLI
6. [T06: Schema Roundtrip Validation](06_schema_roundtrip.ipynb) — offline roundtrip
7. **T07: Data Migration** ← you are here

To dive deeper, explore the API docs at `{BACKEND_URL}/docs` and
`{MIGRATION_URL}/docs`.